# Evaluar el RAG conversacional con LangSmith

En L4 cambiamos el modelo de chat de `gpt-4o-mini` a `claude-haiku-4-5` y lo validamos
**a ojo**, leyendo las respuestas. Así descubrimos por casualidad que la reformulación
había empeorado: Haiku *contestaba* la pregunta (inventando calorías) en vez de reescribirla.

Este notebook convierte esa validación manual en una **medición repetible**.

## Objetivos

1. **Trazar** cada llamada del RAG en LangSmith, sin tocar el código de las cadenas.
2. **Construir un dataset** de preguntas con respuesta de referencia, incluyendo seguimientos
   que dependen del historial y preguntas cuya respuesta no está en los PDFs.
3. **Evaluar** con jueces LLM (`openevals`): corrección, fidelidad al contexto,
   relevancia de lo recuperado y utilidad de la respuesta.
4. **Medir la reformulación** con un evaluador propio, que es justo lo que falló en L4.
5. **Comparar experimentos**: dos modelos × dos formas de pasar el historial al reformulador.

## Por qué no RAGAS

RAGAS era la primera opción, pero su última versión (0.4.3) no es compatible con LangChain 1.x:
importa `langchain_community.chat_models.vertexai`, que ya no existe, y al instalarse
baja `openai` y `langchain-openai` a versiones viejas. `openevals` es la librería de evaluadores
de LangChain, cubre métricas equivalentes y se integra de forma nativa con LangSmith.

| RAGAS | openevals | Qué pregunta |
|---|---|---|
| faithfulness | `RAG_GROUNDEDNESS_PROMPT` | ¿la respuesta se apoya en los chunks o inventa? |
| context precision | `RAG_RETRIEVAL_RELEVANCE_PROMPT` | ¿el retriever trajo algo útil? |
| answer relevancy | `RAG_HELPFULNESS_PROMPT` | ¿la respuesta contesta lo que se preguntó? |
| answer correctness | `CORRECTNESS_PROMPT` | ¿coincide con la respuesta de referencia? |
| – | evaluador propio | ¿la búsqueda usó una pregunta autónoma, o una respuesta? |

## Cómo funciona

```mermaid
flowchart TB
    subgraph DS["Dataset en LangSmith"]
        EX["ejemplo<br/>history · question · reference"]
    end

    subgraph EXP["Experimento · uno por combinación modelo × reformulador"]
        direction TB
        EX --> T["target(inputs)<br/>reproduce el historial turno a turno<br/>y hace la pregunta final"]
        T --> OUT["outputs<br/>answer · query · contexts"]
    end

    subgraph JUD["Jueces · gpt-5.4-mini"]
        direction LR
        OUT --> J1[correctness]
        OUT --> J2[groundedness]
        OUT --> J3[retrieval_relevance]
        OUT --> J4[helpfulness]
        OUT --> J5["rewrite_quality<br/>solo si hay historial"]
    end

    J1 & J2 & J3 & J4 & J5 --> FB[("feedback<br/>en LangSmith")]
    FB --> CMP["tabla comparativa<br/>+ vista Compare de LangSmith"]
```

Dos decisiones que conviene tener presentes:

- **El juez es un modelo distinto y más reciente** que los evaluados (`gpt-5.4-mini`), y barato,
  porque cada experimento lo llama decenas de veces. Aun así, al ser de la misma familia que
  `gpt-4o-mini` puede tener cierto sesgo a su favor: los números sirven para comparar
  experimentos entre sí, no como verdad absoluta.
- **El historial se reproduce con el modelo evaluado.** Las respuestas de los turnos previos
  no están fijadas en el dataset, así que cada modelo arrastra sus propios errores, igual que
  en una conversación real.


## 1 · Setup

`LANGSMITH_TRACING=true` activa el trazado de todo lo que corre sobre LangChain. Lo ponemos aquí
en vez de en el `.env` para que los demás notebooks no manden trazas sin querer.


In [ ]:
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tracers.langchain import wait_for_all_tracers
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langsmith import Client
from openevals.llm import create_llm_as_judge
from openevals.prompts import (
    CORRECTNESS_PROMPT,
    RAG_GROUNDEDNESS_PROMPT,
    RAG_HELPFULNESS_PROMPT,
    RAG_RETRIEVAL_RELEVANCE_PROMPT,
)

In [ ]:
# find_dotenv sube por las carpetas hasta encontrar el .env de la raíz
load_dotenv(find_dotenv())

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "AdvancedGenAI-L5"

ls_client = Client()

EMBEDDINGS = OpenAIEmbeddings(model="text-embedding-3-small")
DATA_DIR = Path("data/pdfs")  # los PDFs que descargó L4

## 2 · El RAG de L4

Lo reconstruimos igual que en L4: mismos PDFs, mismo troceado, mismo vector store.
Si `data/pdfs/` está vacío, ejecuta antes la sección 2 de L4 para descargarlos.


In [ ]:
docs = DirectoryLoader(str(DATA_DIR), glob="**/*.pdf", loader_cls=PyPDFLoader).load()
chunks = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100).split_documents(docs)
vector_store = Chroma.from_documents(documents=chunks, embedding=EMBEDDINGS)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print(len(docs), "páginas →", len(chunks), "chunks")

### Dos formas de reformular

Para poder **medir** la regresión de L4, el reformulador admite los dos modos:

- `"messages"`: el historial va como conversación (`MessagesPlaceholder`). Es el prompt original de L4,
  el que funcionaba con `gpt-4o-mini` y fallaba con Claude.
- `"text"`: el historial va como texto dentro de un único mensaje. Es el arreglo actual de L4.


In [ ]:
contextualize_system_prompt = (
    "Dado el historial de chat y la última pregunta del usuario, reformúlala "
    "como una pregunta autónoma que se entienda sin el historial. "
    "NO respondas la pregunta, solo reformúlala si hace falta."
)

CONTEXTUALIZE_PROMPTS = {
    "messages": ChatPromptTemplate.from_messages([
        ("system", contextualize_system_prompt),
        MessagesPlaceholder("messages"),
    ]),
    "text": ChatPromptTemplate.from_messages([
        ("system", contextualize_system_prompt),
        ("human",
         "<historial>\n{historial}\n</historial>\n\n"
         "<pregunta>{pregunta}</pregunta>\n\n"
         "Devuelve SOLO la pregunta reformulada, sin responderla ni añadir nada más."),
    ]),
}

system_prompt = (
    "Eres un asistente que responde sobre múltiples PDFs. Incluye emojis en cada "
    "respuesta y cita el documento del que sacas la información. "
    "Si la respuesta no está en el contexto, dilo.\n\nContexto:\n{contexto}"
)
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("messages"),
])

In [ ]:
def format_docs(docs):
    """Junta los chunks recuperados en un solo string citable."""
    return "\n\n".join(
        f"[{Path(d.metadata['source']).name} p.{d.metadata.get('page', 0) + 1}]\n{d.page_content}"
        for d in docs
    )


def format_history(mensajes):
    """Convierte el historial en texto plano."""
    return "\n".join(
        f"{'Usuario' if m.type == 'human' else 'Asistente'}: {m.content}"
        for m in mensajes
    )


def build_rag(llm, rewrite_mode: str):
    """Devuelve una función ask(pregunta, chat_history) con el modelo y reformulador elegidos."""
    contextualize_chain = CONTEXTUALIZE_PROMPTS[rewrite_mode] | llm | StrOutputParser()
    qa_chain = qa_prompt | llm | StrOutputParser()

    def ask(pregunta: str, chat_history: list) -> dict:
        mensajes = chat_history + [HumanMessage(pregunta)]

        if not chat_history:
            query = pregunta
        elif rewrite_mode == "messages":
            query = contextualize_chain.invoke({"messages": mensajes})
        else:
            query = contextualize_chain.invoke({
                "historial": format_history(chat_history),
                "pregunta": pregunta,
            })

        found = retriever.invoke(query)
        respuesta = qa_chain.invoke({"contexto": format_docs(found), "messages": mensajes})

        chat_history.extend([HumanMessage(pregunta), AIMessage(respuesta)])
        return {"answer": respuesta, "query": query, "contexts": format_docs(found)}

    return ask

## 3 · El dataset

Diez ejemplos que cubren los cuatro PDFs. Cada uno tiene:

- `history`: preguntas previas (vacío en las de un solo turno). Solo guardamos las **preguntas**;
  las respuestas las genera el modelo evaluado.
- `question`: la pregunta que se evalúa.
- `reference`: la respuesta correcta, sacada a mano de los PDFs.

| Tipo | Cuántos | Qué pone a prueba |
|---|---|---|
| Un turno | 4 | recuperación y respuesta básicas |
| Seguimiento | 6 | la reformulación: *"¿y cuál es la más económica?"* |

Tres de los diez (Alemania, calorías y precio del metro) **no tienen respuesta en los PDFs**:
comprueban que el modelo diga "no está" en vez de inventar. El de las calorías es exactamente
el caso que falló en L4.


In [ ]:
EXAMPLES = [
    # Un turno
    {"history": [],
     "question": "¿Cuáles fueron las ventas totales de la Tienda ABC en el último trimestre?",
     "reference": "$150,000, un 15% más que el trimestre anterior, con 2,500 transacciones."},
    {"history": [],
     "question": "¿Qué producto generó más ingresos en la Tienda ABC?",
     "reference": "El Reloj de Pulsera, con $20,000 en ingresos (200 unidades)."},
    {"history": [],
     "question": "¿Qué desafíos enfrenta la IA en la medicina?",
     "reference": "Privacidad de los datos, falta de regulaciones adecuadas y poca confianza de los médicos en los algoritmos."},
    {"history": [],
     "question": "¿Cuál es la capital de Alemania?",
     "reference": "Los documentos no contienen esa información.",
     "answerable": False},
    # Seguimientos que dependen del historial
    {"history": ["¿Qué platos tiene el recetario?"],
     "question": "¿y cuántas calorías tienen?",
     "reference": "El recetario no incluye información sobre calorías.",
     "answerable": False},
    {"history": ["¿Qué ingredientes lleva el salmón al horno?"],
     "question": "¿a qué temperatura y cuánto tiempo se hornea?",
     "reference": "A 180°C durante 20 minutos."},
    {"history": ["¿Dónde puedo alojarme en París?"],
     "question": "¿y cuál es la opción más económica?",
     "reference": "El Generator Hostel, un hostel moderno pensado para jóvenes viajeros."},
    {"history": ["¿Qué categorías de productos analiza el informe de la Tienda ABC?"],
     "question": "¿cuál aportó más ventas y cuánto?",
     "reference": "Ropa, con el 40% de las ventas totales, es decir $60,000."},
    {"history": ["¿Qué aplicaciones actuales tiene la IA en medicina?", "¿y qué dice sobre su futuro?"],
     "question": "¿cuál de esas ayudaría a reducir los tiempos de espera?",
     "reference": "La optimización de sistemas de salud: los hospitales usarán IA para gestionar recursos, optimizar el flujo de pacientes y reducir los tiempos de espera."},
    {"history": ["¿Qué medios de transporte recomienda la guía de París?"],
     "question": "¿cuánto cuesta el billete de metro?",
     "reference": "La guía no indica el precio del billete de metro; solo dice que el metro tiene 16 líneas.",
     "answerable": False},
]

### Splits y metadatos

Cada ejemplo se guarda en un **split** (`single_turn` o `followup`) y con `answerable` en sus
metadatos. Los splits son una función de LangSmith: en la UI puedes filtrar las métricas por split
y ver, por ejemplo, que un modelo solo falla en los seguimientos. Además permiten evaluar contra
un subconjunto (`data=ls_client.list_examples(..., splits=["followup"])`) cuando no quieres pagar
el dataset entero.

Creamos el dataset **solo si no existe**, para que re-ejecutar el notebook no duplique ejemplos.
Si cambias `EXAMPLES`, sube el número de `DATASET_NAME`: los experimentos viejos siguen colgando
del dataset viejo y no se mezclan con los nuevos.


In [ ]:
DATASET_NAME = "L5-rag-conversational-v4"

if ls_client.has_dataset(dataset_name=DATASET_NAME):
    print("Ya existe:", DATASET_NAME)
else:
    dataset = ls_client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Preguntas sobre los 4 PDFs de L4, con seguimientos y preguntas sin respuesta.",
    )
    ls_client.create_examples(
        dataset_id=dataset.id,
        examples=[
            {
                "inputs": {"history": e["history"], "question": e["question"]},
                "outputs": {"reference": e["reference"]},
                "split": "followup" if e["history"] else "single_turn",
                "metadata": {"answerable": e.get("answerable", True)},
            }
            for e in EXAMPLES
        ],
    )
    print("Creado:", DATASET_NAME, "con", len(EXAMPLES), "ejemplos")

## 4 · Los evaluadores

`create_llm_as_judge` convierte un prompt en un evaluador: el juez lee el prompt relleno,
razona y devuelve una nota con un comentario que la explica.

Usamos dos escalas según lo que mide cada evaluador:

| Evaluador | Escala | Por qué |
|---|---|---|
| `correctness`, `retrieval_relevance`, `helpfulness` | `0 · 0.5 · 1` | "parcialmente" tiene sentido: una respuesta puede acertar la temperatura y olvidar el tiempo |
| `groundedness`, `rewrite_quality` | `true · false` | son sí/no: la respuesta inventa o no, la consulta es una pregunta o no |

No usamos una escala continua (`continuous=True`): a un LLM le cuesta calibrar un número, y el
mismo caso puede sacar 0.7 u 0.85 en dos ejecuciones. Pocas opciones dan notas más estables.

Los prompts de `openevals` esperan variables concretas (`inputs`, `outputs`, `context`,
`reference_outputs`), así que cada evaluador es un pequeño adaptador que saca esos campos
de lo que devolvió el RAG.

En `inputs` pasamos **la conversación completa**, con las respuestas previas del asistente
(`outputs["conversation"]`, que construye el `target`). Sin ella, el juez no sabría qué significa
*"¿y cuál es la más económica?"*, ni de dónde salen los nombres que aparezcan en la respuesta.


In [ ]:
JUDGE = ChatOpenAI(model="gpt-5.4-mini")
SCALE = [0.0, 0.5, 1.0]  # fallo · parcial · correcto


correctness_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT, judge=JUDGE, feedback_key="correctness", choices=SCALE)
groundedness_judge = create_llm_as_judge(
    prompt=RAG_GROUNDEDNESS_PROMPT, judge=JUDGE, feedback_key="groundedness")
retrieval_judge = create_llm_as_judge(
    prompt=RAG_RETRIEVAL_RELEVANCE_PROMPT, judge=JUDGE, feedback_key="retrieval_relevance",
    choices=SCALE)
helpfulness_judge = create_llm_as_judge(
    prompt=RAG_HELPFULNESS_PROMPT, judge=JUDGE, feedback_key="helpfulness", choices=SCALE)


def correctness(inputs, outputs, reference_outputs):
    return correctness_judge(
        inputs=outputs["conversation"],
        outputs=outputs["answer"],
        reference_outputs=reference_outputs["reference"],
    )


def groundedness(inputs, outputs):
    return groundedness_judge(context=outputs["contexts"], outputs=outputs["answer"])


def retrieval_relevance(inputs, outputs):
    return retrieval_judge(inputs=outputs["conversation"], context=outputs["contexts"])


def helpfulness(inputs, outputs):
    return helpfulness_judge(inputs=outputs["conversation"], outputs=outputs["answer"])

### El evaluador propio: calidad de la reformulación

Ninguna métrica estándar mira el paso intermedio. `groundedness` y `correctness` solo ven
la respuesta final, que en L4 salió bien *a pesar* de que la búsqueda se hizo con texto inventado.

Este evaluador mira `query`, lo que realmente se buscó en Chroma, y solo se aplica
a los ejemplos con historial (sin historial no hay nada que reformular).

El segundo evaluador propio, `abstention`, solo corre sobre los 3 ejemplos marcados
`answerable=False` y comprueba lo contrario de lo habitual: que el modelo **no** conteste.
Un evaluador puede leer los metadatos del ejemplo (`example.metadata`) para decidir si aplica o no;
devolver `{"results": []}` significa "esta métrica no aplica aquí", y LangSmith no la cuenta.


In [ ]:
REWRITE_PROMPT = """Eres un evaluador experto de sistemas RAG conversacionales.

Un sistema recibió una conversación y tenía que convertir la última pregunta en una
consulta de búsqueda autónoma, SIN responderla.

<conversacion>
{inputs}
</conversacion>

<consulta_generada>
{outputs}
</consulta_generada>

La consulta es correcta (true) solo si cumple TODO lo siguiente:
- Es una pregunta o consulta de búsqueda, no una respuesta ni una explicación.
- No contiene datos que no aparezcan en la conversación (nada inventado).
- Se entiende sin leer el historial: resuelve referencias como "esas", "¿y...?", "la más económica".
- Conserva la intención de la última pregunta del usuario.

En cualquier otro caso es false."""

rewrite_judge = create_llm_as_judge(
    prompt=REWRITE_PROMPT, judge=JUDGE, feedback_key="rewrite_quality")


def rewrite_quality(inputs, outputs):
    if not inputs["history"]:
        return {"results": []}  # sin historial no hay reformulación que evaluar
    return rewrite_judge(inputs=outputs["conversation"], outputs=outputs["query"])


ABSTENTION_PROMPT = """Eres un evaluador de sistemas RAG.

La siguiente pregunta NO tiene respuesta en los documentos indexados.

<conversacion>
{inputs}
</conversacion>

<respuesta>
{outputs}
</respuesta>

La respuesta es correcta (true) solo si reconoce con claridad que la información no está en los
documentos y NO aporta el dato como si viniera de ellos. Si lo inventa, o lo responde con
conocimiento propio sin dejar claro que no sale de los documentos, es false."""

abstention_judge = create_llm_as_judge(
    prompt=ABSTENTION_PROMPT, judge=JUDGE, feedback_key="abstention")


def abstention(inputs, outputs, example):
    # los argumentos válidos son run, example, inputs, outputs, reference_outputs y attachments
    if (example.metadata or {}).get("answerable", True):
        return {"results": []}  # solo aplica a las preguntas sin respuesta en los PDFs
    return abstention_judge(inputs=outputs["conversation"], outputs=outputs["answer"])


EVALUATORS = [correctness, groundedness, retrieval_relevance, helpfulness,
              rewrite_quality, abstention]

## 5 · Dos experimentos, una pregunta cada uno

Hay dos decisiones pendientes: **qué modelo** y **cómo pasarle el historial al reformulador**.
Medirlas a la vez (2 × 2) responde las dos de golpe, pero cuesta el doble y cuesta más de leer.
Aquí van en cadena: **el ganador del experimento 1 es el que entra al experimento 2**.

| | Pregunta | Qué varía | Qué se fija |
|---|---|---|---|
| **E1** | ¿cómo pasar el historial al reformulador? | `messages` vs `text` | modelo `gpt-4o-mini` |
| **E2** | ¿qué modelo? | `gpt-4o-mini` vs `claude-haiku-4-5` | el reformulador ganador de E1 |

Se empieza por el prompt con el modelo de partida fijo, que es la única configuración que no
presupone nada, y el modelo se decide después sobre el mejor prompt.

> ⚠️ **Encadenar asume que las dos variables no interactúan**, y eso no está garantizado: un prompt
> puede convenirle a un modelo y no a otro. La comprobación es barata: si en E2 gana el otro modelo,
> se repite E1 con él y se mira si el reformulador ganador sigue siendo el mismo.

### Cómo se nombran los experimentos

Dos experimentos solo son comparables si se midieron **con la misma vara**: mismo dataset,
mismos jueces y misma escala de notas. Cambiar el juez y no cambiar el nombre deja la lista
llena de experimentos que parecen comparables y no lo son.

```
L5.r4.e1.text
│  │  │  └── variante dentro del experimento (el modo en E1, el modelo en E2)
│  │  └───── experimento: e1 = reformulador, e2 = modelo
│  └──────── RUN_TAG: la "vara" de medición
└─────────── notebook
```

**`RUN_TAG` sube cada vez que cambia algo que afecta a las notas**: el juez, la escala, los prompts
de los evaluadores o el dataset. Regla de lectura: **solo se comparan entre sí los experimentos con
el mismo `RUN_TAG`**, y dentro de él, los del mismo `eN`. Como el nombre empieza por el notebook y
sigue por el run, LangSmith los ordena juntos alfabéticamente.

Los mismos datos van también en `metadata`, que es lo que permite filtrar en la UI
(*Filter → Metadata*) sin depender del nombre.

### Repeticiones

`num_repetitions=2` corre cada ejemplo dos veces. Con 10 ejemplos, un solo caso mueve la media
10 puntos, y los jueces LLM no son deterministas: si una métrica baila entre repeticiones,
una diferencia de 5 puntos entre dos experimentos no significa nada.

> ⚠️ Cuesta dinero: 4 experimentos (2 por cada pregunta) × 10 ejemplos × 2 repeticiones, con
> 1 a 7 llamadas al modelo evaluado y hasta 6 al juez por ejecución.


### Métricas de todo el experimento

Un `summary_evaluator` recibe **todas** las ejecuciones de golpe y devuelve una sola nota por
experimento. Sirve para lo que no es una nota por ejemplo: aquí, la latencia media, que aparece
como una columna más junto a las métricas de calidad.

> El costo **no** se puede calcular aquí: los objetos que llegan son `RunTree`, que no tienen
> `total_cost`. LangSmith calcula el costo en su servidor *después* de recibir los runs, así que
> hay que pedirlo luego (sección 7).


In [ ]:
def avg_latency_s(runs, examples) -> dict:
    """Segundos medios por pregunta, incluidos los turnos previos del historial."""
    times = [(r.end_time - r.start_time).total_seconds() for r in runs if r.end_time]
    return {"key": "avg_latency_s", "score": sum(times) / len(times) if times else None}


SUMMARY_EVALUATORS = [avg_latency_s]

In [ ]:
RUN_TAG = "r4"  # súbelo si cambias juez, escala, prompts de evaluadores o dataset

MODELS = {
    # código corto para el nombre del experimento: modelo
    "gpt4omini": ChatOpenAI(model="gpt-4o-mini", temperature=0, max_completion_tokens=400),
    "haiku45": ChatAnthropic(model="claude-haiku-4-5", temperature=0, max_tokens=400),
}


def make_target(llm, rewrite_mode: str):
    ask = build_rag(llm, rewrite_mode)

    def target(inputs: dict) -> dict:
        chat_history = []  # historial nuevo por ejemplo: los ejemplos no se contaminan
        for previa in inputs["history"]:
            ask(previa, chat_history)

        previas = format_history(chat_history)  # antes de la pregunta final
        result = ask(inputs["question"], chat_history)
        result["conversation"] = (
            (f"{previas}\n" if previas else "")
            + f"Usuario (pregunta a evaluar): {inputs['question']}"
        )
        return result

    return target


def run_experiment(exp: str, variante: str, model_code: str, rewrite_mode: str):
    """Lanza un experimento con el nombre y los metadatos de la convención."""
    nombre = f"L5.{RUN_TAG}.{exp}.{variante}"
    print("▶", nombre)
    return ls_client.evaluate(
        make_target(MODELS[model_code], rewrite_mode),
        data=DATASET_NAME,
        evaluators=EVALUATORS,
        summary_evaluators=SUMMARY_EVALUATORS,
        experiment_prefix=nombre,
        metadata={
            "notebook": "L5",
            "run_tag": RUN_TAG,
            "experimento": exp,
            "variante": variante,
            "model": model_code,
            "rewrite_mode": rewrite_mode,
            "judge": "gpt-5.4-mini",
            "scale": "mixta: 0/0.5/1 + booleana",
            "dataset": DATASET_NAME,
        },
        num_repetitions=2,
        max_concurrency=4,
    )

### Estimar antes de gastar

Antes de lanzar nada conviene saber cuánto va a costar. La estimación se hace con un **piloto**:
se corre el pipeline completo (RAG + jueces) sobre **dos ejemplos**, uno de cada tipo, se miden
los tokens reales y se extrapola al dataset entero.

`get_usage_metadata_callback()` captura los tokens de **todas** las llamadas a modelos que ocurran
dentro del bloque `with`, agrupados por modelo. Con eso y una tabla de precios sale el costo.

> Los precios están consultados el **18-09-2026** en
> [openai](https://developers.openai.com/api/docs/pricing) y
> [anthropic](https://platform.claude.com/docs/en/about-claude/pricing). Cámbialos si han variado:
> es la parte de este cálculo que caduca.
>
> Quedan fuera los embeddings de las búsquedas ($0.02 por millón de tokens), que el callback no
> captura y que a esta escala son céntimos.


In [ ]:
from langchain_core.callbacks import get_usage_metadata_callback

PRECIOS_USD_POR_MTOK = {           # (entrada, salida) por millón de tokens
    "gpt-4o-mini": (0.15, 0.60),
    "gpt-5.4-mini": (0.75, 4.50),  # el juez
    "claude-haiku-4-5": (1.00, 5.00),
}


def precio(modelo: str) -> tuple[float, float]:
    """Los proveedores devuelven nombres con fecha (gpt-4o-mini-2024-07-18): busca por prefijo."""
    for nombre, tarifa in PRECIOS_USD_POR_MTOK.items():
        if modelo.startswith(nombre):
            return tarifa
    raise KeyError(f"añade el precio de {modelo} a PRECIOS_USD_POR_MTOK")


def costo_usd(usage: dict, solo_juez: bool = False) -> float:
    total = 0.0
    for modelo, u in usage.items():
        es_juez = modelo.startswith("gpt-5.4-mini")
        if es_juez != solo_juez:
            continue
        entrada, salida = precio(modelo)
        total += u["input_tokens"] / 1e6 * entrada + u["output_tokens"] / 1e6 * salida
    return total

In [ ]:
def piloto(model_code: str, rewrite_mode: str, ejemplo: dict) -> dict:
    """Corre un ejemplo por el pipeline completo y devuelve lo que costó, en dólares."""
    target = make_target(MODELS[model_code], rewrite_mode)

    with get_usage_metadata_callback() as cb:
        outputs = target({"history": ejemplo["history"], "question": ejemplo["question"]})

        correctness_judge(inputs=outputs["conversation"], outputs=outputs["answer"],
                          reference_outputs=ejemplo["reference"])
        groundedness_judge(context=outputs["contexts"], outputs=outputs["answer"])
        retrieval_judge(inputs=outputs["conversation"], context=outputs["contexts"])
        helpfulness_judge(inputs=outputs["conversation"], outputs=outputs["answer"])
        if ejemplo["history"]:
            rewrite_judge(inputs=outputs["conversation"], outputs=outputs["query"])
        if not ejemplo.get("answerable", True):
            abstention_judge(inputs=outputs["conversation"], outputs=outputs["answer"])

    return {"modelo": costo_usd(cb.usage_metadata),
            "juez": costo_usd(cb.usage_metadata, solo_juez=True)}


def estimar(model_code: str, rewrite_mode: str, repeticiones: int = 2) -> dict:
    """Extrapola el piloto al dataset entero, pesando cada tipo de pregunta por su frecuencia."""
    muestras = {
        "single_turn": next(e for e in EXAMPLES if not e["history"]),
        "followup": next(e for e in EXAMPLES if e["history"]),
    }
    cuantos = {
        "single_turn": sum(1 for e in EXAMPLES if not e["history"]),
        "followup": sum(1 for e in EXAMPLES if e["history"]),
    }

    total = {"modelo": 0.0, "juez": 0.0}
    for tipo, ejemplo in muestras.items():
        medido = piloto(model_code, rewrite_mode, ejemplo)
        for k in total:
            total[k] += medido[k] * cuantos[tipo] * repeticiones

    total["total"] = total["modelo"] + total["juez"]
    return total

In [ ]:
CONFIGURACIONES = {
    "e1 · messages": ("gpt4omini", "messages"),
    "e1 · text": ("gpt4omini", "text"),
    "e2 · gpt4omini": ("gpt4omini", "text"),
    "e2 · haiku45": ("haiku45", "text"),
}

estimaciones = pd.DataFrame({
    nombre: estimar(model_code, rewrite_mode)
    for nombre, (model_code, rewrite_mode) in CONFIGURACIONES.items()
}).T

print(f"estimación total de los 4 experimentos: ${estimaciones['total'].sum():.2f}")
estimaciones.style.format("${:.4f}")

El piloto asume que `e2 · gpt4omini` usará el reformulador `text`. Si E1 diera otro ganador,
esa fila habría que rehacerla — es el precio de estimar antes de decidir.

### Experimento 1 · ¿cómo pasar el historial?

Mismo RAG y mismo modelo (`gpt-4o-mini`); lo único que cambia es cómo recibe el historial
el prompt de reformulación.


In [ ]:
MODELO_BASE = "gpt4omini"

exp1 = {
    mode: run_experiment("e1", mode, model_code=MODELO_BASE, rewrite_mode=mode)
    for mode in ["messages", "text"]
}

**La regla de decisión se escribe antes de ver los números**, si no, siempre hay una forma de
leer la tabla que confirma lo que uno quería: gana la media de las métricas de calidad, y si dos
quedan a menos de 5 puntos (que es ruido con 10 ejemplos), gana el más barato.


In [ ]:
METRICS = ["correctness", "groundedness", "retrieval_relevance",
           "helpfulness", "rewrite_quality", "abstention"]


def tabla(resultados: dict) -> pd.DataFrame:
    """Media de cada métrica por experimento. NaN = la métrica no aplicaba a ningún ejemplo."""
    return pd.DataFrame({
        nombre: {m: res.to_pandas()[f"feedback.{m}"].astype(float).mean() for m in METRICS}
        for nombre, res in resultados.items()
    }).T


def runs_del_experimento(res, intentos: int = 20, espera: int = 6) -> list:
    """Los runs del RAG, uno por ejemplo y repetición.

    Dos esperas distintas, y las dos hacen falta:

    1. Los runs se suben en segundo plano. El tracer de LangChain tiene su **propia** cola, y
       `ls_client.flush()` no la vacía: hace falta `wait_for_all_tracers()`. Sin eso, los runs
       pueden tardar minutos en aparecer, o no aparecer nunca.
    2. LangSmith les calcula el costo en su servidor, unos segundos después de recibirlos.
    """
    res.wait()
    wait_for_all_tracers()   # la cola del tracer de LangChain (el RAG)
    ls_client.flush()        # la cola de este cliente

    runs = []
    for _ in range(intentos):
        runs = list(ls_client.list_runs(project_id=res.experiment_id, is_root=True))
        if runs and all(r.total_cost is not None for r in runs):
            return runs
        time.sleep(espera)

    if not runs:
        # Pasa sobre todo al agotar la cuota mensual de trazas del plan: LangSmith acepta las
        # peticiones con un 429 y descarta los runs, así que el experimento queda vacío.
        print(f"⚠️  LangSmith no devolvió runs de {res.experiment_name}: "
              f"¿cuota de trazas agotada? Las tablas de costo real saldrán vacías.")
    return runs  # puede venir vacío o con costos incompletos


def costo_medio(res) -> float:
    """Costo real por pregunta. Solo para la sección 7: consulta LangSmith y puede tardar."""
    runs = runs_del_experimento(res)
    return sum(float(r.total_cost or 0) for r in runs) / len(runs)  # float: LangSmith devuelve Decimal


def ganador(resultados: dict, experimento: str, margen: float = 0.05) -> str:
    """Gana la calidad; si la diferencia cabe en el ruido, gana el más barato.

    Para el costo usa la **estimación** del piloto, no LangSmith: justo después de `evaluate`,
    los runs pueden tardar minutos en ser consultables, y la decisión no puede depender de eso.
    El costo real se compara en la sección 7.
    """
    calidad = tabla(resultados).mean(axis=1)          # media de las métricas que aplican
    empatados = calidad[calidad >= calidad.max() - margen].index
    costo = lambda n: estimaciones.loc[f"{experimento} · {n}", "total"]
    return min(empatados, key=lambda n: (round(costo(n), 4), -calidad[n]))


tabla1 = tabla(exp1)
tabla1.style.format("{:.0%}", na_rep="–")

In [ ]:
modo_ganador = ganador(exp1, "e1")

print("calidad media: ", {k: f"{v:.0%}" for k, v in tabla(exp1).mean(axis=1).items()})
print("costo estimado:", {k: f"${estimaciones.loc[f'e1 · {k}', 'total']:.4f}" for k in exp1})
print()
print("ganador de E1:", modo_ganador)

### Experimento 2 · ¿qué modelo?

El reformulador queda fijo en el ganador de E1 y lo único que cambia es el modelo de chat.


In [ ]:
exp2 = {
    code: run_experiment("e2", code, model_code=code, rewrite_mode=modo_ganador)
    for code in MODELS
}

tabla2 = tabla(exp2)
tabla2.style.format("{:.0%}", na_rep="–")

In [ ]:
modelo_ganador = ganador(exp2, "e2")
print("ganador de E2:", modelo_ganador)
print(f"\nconfiguración elegida: {modelo_ganador} + reformulador {modo_ganador}")

## 6 · Lo que hay detrás de los números

Las tres vistas siguientes cubren los cuatro experimentos a la vez. Sirven para no quedarse
con la media, que esconde justo lo que interesa.


In [ ]:
TODOS = {f"e1 · {k}": v for k, v in exp1.items()}
TODOS |= {f"e2 · {k}": v for k, v in exp2.items()}

frames = {nombre: res.to_pandas() for nombre, res in TODOS.items()}

### Calidad por split

La misma tabla separada por tipo de pregunta. Es donde se ve si un fallo viene de los seguimientos
o de las preguntas de un turno, en las que no hay reformulación que pueda fallar.


In [ ]:
por_split = pd.concat({
    nombre: df.assign(split=df["inputs.history"].map(lambda h: "followup" if h else "single_turn"))
              .groupby("split")[[f"feedback.{m}" for m in METRICS]].mean()
    for nombre, df in frames.items()
}, names=["experimento"])
por_split.columns = [c.removeprefix("feedback.") for c in por_split.columns]

por_split.style.format("{:.0%}", na_rep="–")

### Costo por pregunta

Lo que costó cada pregunta del dataset, con su latencia al lado. La calidad sin el costo no permite
decidir: un modelo que acierta 5 puntos más pero cuesta diez veces más puede no compensar, y eso
depende de tu caso, no de la tabla.


In [ ]:
def runs_del_juez(res):
    """Los jueces se trazan aparte, en el proyecto 'evaluators', etiquetados con el experimento."""
    consulta = '{"experiment": "%s"}' % res.experiment_name
    filtro = "has(metadata, '%s')" % consulta
    return list(ls_client.list_runs(project_name="evaluators", filter=filtro, is_root=True))


def media(valores: list, n: int) -> float:
    return sum(valores) / n if n else float("nan")   # sin runs: NaN, no una división por cero


costos = {}
for nombre, res in TODOS.items():
    runs = runs_del_experimento(res)
    costos[nombre] = {
        "costo_medio_usd": media([float(r.total_cost or 0) for r in runs], len(runs)),
        "tokens_medios": media([r.total_tokens or 0 for r in runs], len(runs)),
        "latencia_media_s": media([(r.end_time - r.start_time).total_seconds()
                                   for r in runs if r.end_time], len(runs)),
    }

pd.DataFrame(costos).T.style.format(
    {"costo_medio_usd": "${:.4f}", "tokens_medios": "{:.0f}", "latencia_media_s": "{:.1f} s"},
    na_rep="sin datos")

### ¿Qué se buscó en cada caso?

Los números dicen *cuánto* falla la reformulación; esta tabla enseña *cómo*. Para cada ejemplo
con historial, la consulta que cada experimento mandó a Chroma.


In [ ]:
pd.set_option("display.max_colwidth", 120)

# groupby().first(): con num_repetitions=2 hay dos ejecuciones por pregunta
queries = pd.DataFrame({
    nombre: df.groupby("inputs.question")["outputs.query"].first()
    for nombre, df in frames.items()
})
primero = next(iter(frames.values())).groupby("inputs.question")["inputs.history"].first()
queries.loc[primero[primero.map(len) > 0].index]

## 7 · Estimación contra realidad

Ahora se puede cerrar el círculo: lo que estimamos con el piloto de dos ejemplos, contra lo que
LangSmith registró de verdad.

El costo real tiene dos partes que viven en sitios distintos:

- **el modelo evaluado**, en los runs del experimento;
- **el juez**, en el proyecto `evaluators`, etiquetado con el nombre del experimento en sus metadatos.

Sumar solo la primera parte es el error fácil, y aquí se ve por qué importa: **evaluar cuesta más
que ejecutar**.

> Si estas tablas salen vacías, lo normal es que se haya agotado la **cuota mensual de trazas**
> del plan de LangSmith: responde `429` y descarta los runs, así que el experimento existe pero
> no tiene datos. Los experimentos siguen ejecutándose y costando dinero en los proveedores,
> pero no quedan registrados. Es la razón de peso para estimar antes y no re-ejecutar a lo tonto. Los jueces leen la conversación, el contexto recuperado y la respuesta, y razonan
antes de dar la nota.


In [ ]:
real = {}
for nombre, res in TODOS.items():
    real[nombre] = {
        "real_modelo": sum(float(r.total_cost or 0) for r in runs_del_experimento(res)),
        "real_juez": sum(float(r.total_cost or 0) for r in runs_del_juez(res)),
    }

costos_reales = comparacion = pd.DataFrame(real).T
comparacion["real_total"] = comparacion["real_modelo"] + comparacion["real_juez"]
comparacion["estimado"] = estimaciones["total"]
comparacion["desviación"] = comparacion["estimado"] / comparacion["real_total"] - 1

if comparacion["real_total"].sum() == 0:
    print("⚠️  sin costos reales: LangSmith no tiene runs de estos experimentos "
          "(cuota de trazas agotada). Vuelve a ejecutar cuando se reinicie.")
else:
    print(f"estimado: ${comparacion['estimado'].sum():.2f}   "
          f"real: ${comparacion['real_total'].sum():.2f}")

comparacion[["estimado", "real_modelo", "real_juez", "real_total", "desviación"]].style.format(
    {"estimado": "${:.4f}", "real_modelo": "${:.4f}", "real_juez": "${:.4f}",
     "real_total": "${:.4f}", "desviación": "{:+.0%}"})

Una desviación de ±20% es normal y la estimación sigue sirviendo: lo que se decide con ella
es si lanzar el experimento o no, no cuánto facturar. Las fuentes de error son siempre las mismas:

- **dos ejemplos no representan a diez**: la longitud de la respuesta varía, y con ella los tokens de salida;
- **los reintentos** de un juez que devuelve algo mal formado se pagan y no estaban en el piloto;
- **los turnos previos** de los seguimientos dependen de cuánto se enrolle el modelo en cada uno.

Si la desviación fuera mucho mayor, el piloto está mal elegido: usa más ejemplos, o los más largos.


## 8 · Conclusiones

La media decide, pero esconde el detalle: dos configuraciones pueden empatar en promedio y ganar
cada una en métricas distintas. El marcador desglosa **quién ganó, empató o perdió en cada métrica**.

Una diferencia cuenta como victoria solo si supera el **margen de ruido** (5 puntos, el mismo de la
regla de decisión). Con 10 ejemplos y 2 repeticiones, por debajo de eso no se distingue nada:
un único ejemplo que cambie de nota mueve la métrica 10 puntos.


In [ ]:
def marcador(resultados: dict, margen: float = 0.05) -> pd.DataFrame:
    """Compara dos variantes métrica a métrica: quién gana, quién pierde y dónde hay empate."""
    t = tabla(resultados)
    a, b = t.index

    filas = {}
    for m in METRICS:
        va, vb = t.loc[a, m], t.loc[b, m]
        if pd.isna(va) and pd.isna(vb):
            continue  # métrica que no aplicaba a ningún ejemplo
        diferencia = va - vb
        if abs(diferencia) < margen:
            veredicto = "empate"
        else:
            veredicto = f"gana {a if diferencia > 0 else b}"
        filas[m] = {a: va, b: vb, "diferencia": diferencia, "veredicto": veredicto}

    return pd.DataFrame(filas).T


def resumen(resultados: dict, experimento: str, campeon: str, margen: float = 0.05) -> None:
    """Imprime cuántas métricas ganó, empató y perdió el ganador, y qué costó cada variante."""
    m = marcador(resultados, margen)
    ganadas = (m["veredicto"] == f"gana {campeon}").sum()
    empatadas = (m["veredicto"] == "empate").sum()
    perdidas = len(m) - ganadas - empatadas

    print(f"{campeon}: {ganadas} ganadas, {empatadas} empatadas, {perdidas} perdidas "
          f"de {len(m)} métricas")
    for nombre in resultados:
        estimado = estimaciones.loc[f"{experimento} · {nombre}", "total"]
        print(f"   {nombre}: ${estimado:.4f} estimados "
              f"({len(EXAMPLES)} preguntas × 2 repeticiones)")

### Experimento 1 · `messages` contra `text`


In [ ]:
resumen(exp1, "e1", modo_ganador)
marcador(exp1).style.format({k: "{:.0%}" for k in list(tabla(exp1).index) + ["diferencia"]})

### Experimento 2 · `gpt-4o-mini` contra `claude-haiku-4-5`


In [ ]:
resumen(exp2, "e2", modelo_ganador)
marcador(exp2).style.format({k: "{:.0%}" for k in list(tabla(exp2).index) + ["diferencia"]})

### El marcador de la corrida `r4`

Estos son los números que quedaron registrados en LangSmith el **20-09-2026**, con el dataset
`L5-rag-conversational-v4`, 10 preguntas × 2 repeticiones por variante. Las celdas de arriba los
recalculan al volver a ejecutar los experimentos; quedan aquí para poder leer el notebook sin
lanzarlos otra vez.

**Experimento 1 · `messages` contra `text`** — *gana `messages`: 2 ganadas, 4 empatadas, 0 perdidas.*

| métrica | `messages` | `text` | diferencia | veredicto |
|---|---:|---:|---:|---|
| correctness | 85% | 85% | 0 | empate |
| groundedness | 95% | 90% | +5 | empate (justo en el margen) |
| retrieval_relevance | 80% | 80% | 0 | empate |
| helpfulness | 88% | 88% | 0 | empate |
| rewrite_quality | 100% | 92% | +8 | gana `messages` |
| abstention | 100% | 83% | +17 | gana `messages` |
| **media** | **91%** | **86%** | +5 | |
| costo | $0.114 | $0.116 | | |

Con `gpt-4o-mini` los dos formatos son casi indistinguibles: cuatro de las seis métricas empatan y
la media se queda **justo en el margen de ruido**. `messages` gana por dos métricas que dependen
del historial, que es exactamente donde el formato del prompt podía notarse.

**Experimento 2 · `gpt-4o-mini` contra `claude-haiku-4-5`** — *gana `gpt4omini`: 2 ganadas, 2 empatadas, 2 perdidas.*

| métrica | `gpt4omini` | `haiku45` | diferencia | veredicto |
|---|---:|---:|---:|---|
| correctness | 88% | 100% | −13 | gana `haiku45` |
| groundedness | 95% | 65% | +30 | gana `gpt4omini` |
| retrieval_relevance | 78% | 80% | −3 | empate |
| helpfulness | 85% | 98% | −13 | gana `haiku45` |
| rewrite_quality | 100% | 0% | +100 | gana `gpt4omini` |
| abstention | 100% | 100% | 0 | empate |
| **media** | **91%** | **74%** | +17 | |
| costo | $0.113 | $0.192 | | |
| latencia media | 3.3 s | 4.9 s | | |

Este es el caso del punto 2 de la lista de abajo: **no hay un ganador, hay dos perfiles**. Haiku
redacta mejor (correctness y helpfulness por encima) pero suspende `rewrite_quality` con un **0%**
redondo, y arrastra `groundedness` a 65%. Las dos cosas son el mismo fallo: con el historial en
formato `messages` Haiku *responde* la pregunta en vez de reescribirla, así que la búsqueda se hace
con una consulta inventada y la respuesta se apoya en chunks que no vienen a cuento. Es el mismo
problema que encontramos a ojo en L4, ahora con un número al lado.

La consecuencia práctica es que **el ganador de E1 no es neutral para E2**: `messages` empataba con
`text` en `gpt-4o-mini`, pero rompe a Haiku. Encadenar experimentos ahorra ejecuciones, y el precio
es este — la decisión de E1 se tomó con datos de un solo modelo. Si el candidato real fuera Haiku,
E1 habría que repetirlo con él.

Y un dato que no estaba en la pregunta de ninguno de los dos experimentos: de los $0.114 de E1,
**$0.108 son de los jueces** y $0.006 del RAG. Evaluar cuesta unas 17 veces más que ejecutar.

### Qué hacer con esto

Tres lecturas, en orden de lo que más cambia una decisión:

1. **Si el ganador gana pocas métricas y empata el resto**, la decisión la está tomando el costo,
   no la calidad. Es una decisión legítima, pero conviene decirlo en voz alta.
2. **Si cada variante gana métricas distintas**, no hay un ganador: hay dos perfiles. Ahí toca
   elegir qué métrica importa más *para tu caso* — no es una pregunta que resuelva la tabla.
3. **Si casi todo es empate**, el dataset no distingue. La solución no es mirar más la tabla, es
   ampliar el dataset o subir `num_repetitions` hasta que las diferencias salgan del ruido.

Y el recordatorio de siempre: antes de fiarse de una métrica, abrir dos o tres comentarios del juez
en LangSmith y comprobar que está midiendo lo que crees que mide.


### Cómo seguir en LangSmith

Todo lo anterior también está en la UI, con más detalle:

- **Datasets & Experiments → `L5-rag-conversational-v4`**: los experimentos, uno por fila, con una
  columna por métrica (incluidos `avg_cost_usd` y `avg_latency_s`). **La comparación no está dentro
  de un experimento**: hay que marcar la casilla de los que quieras y pulsar **Compare**, arriba.
  Entonces cada fila es una pregunta y cada columna un experimento.
- **Compara solo experimentos del mismo `RUN_TAG` y el mismo `eN`**: `L5.r4.e1.*` entre sí,
  `L5.r4.e2.*` entre sí. Con otro `RUN_TAG` cambió la vara de medir, aunque la UI los liste juntos.
  E1 y E2 tampoco se comparan entre sí: en E2 cambia el modelo, no el prompt.
- **En la vista Compare**, el desplegable *Display* deja ver una métrica a la vez, y
  *Filter → Metadata* permite quedarse, por ejemplo, con `experimento = e2`.
- **El comentario del juez** (clic en una nota) explica por qué aprobó o suspendió. Es lo primero
  que hay que leer antes de fiarse de una métrica: un juez también se equivoca.
- **La traza de cada ejemplo** muestra la reformulación, los chunks recuperados y la respuesta,
  con tokens y latencia de cada llamada.

Con el dataset creado, cualquier cambio futuro (otro `chunk_size`, otro prompt, otro modelo) es
un experimento más con el mismo `RUN_TAG`, en lugar de leer respuestas a mano.
